In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('.'))

In [10]:
# Importing stuff for running
from Sim import Holdings # This is neeed to track the bought stocks and the price
from Sim.runner import Runner # This will run the simulations
 # If algorithm needs some set of metric to be calculated this guy will do it
from Sim.metric_calculator import MetricCalculator
from Sim.trend_iter import TrendIter # This will be responsible to recive the stock market price
# This will take care of updating the holdings,
# maybe in future will also perform the trade on the market
from Sim.action_resolver import action_resolver 

In [5]:
# Stuff to be simulated with
from datetime import date
# we are going with ANET, and we will use data from April 14 2026 
data_iter = TrendIter( "ANET", date=date(month=4, day=14, year=2026) ) 


from Algorithm.MACDStrategy import MACDStrategy
# For the sake of it we will go with this algorithm
algorithm = MACDStrategy()

mc = MetricCalculator()

# we are adding metrics needed by algorithm to metric calculator
mc.addMetrics(algorithm.getAlgoMetrics())

c:\Users\Meowmaster\Desktop\PROject\IntradaySimulator\Sim\data_fetcher.py:14: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yfinance.download( tick, interval = interval.value,


In [6]:
# Starting amount ( Note there is no currency unit here I kinda forgot,
#  this will be the currency that whatever the stock is being traded )
funds = 10_000 

holdings = Holdings( quantity=0, avg_price=0 ) # We will start with zero stock in our name

In [8]:
# we have initialized our runner with all the stuff
runner = Runner(funds=funds, positions=holdings, algorithm=algorithm, metric_calculator=mc)

In [11]:
runner.run(data_iter=data_iter, action_resolver=action_resolver )

In [13]:
# This completes the simulation
# Now we will see if we made any profits
float(runner.getFunds())

9997.658584594727

In [14]:
# If we have used the algorithm which doesnt take into consideration 
# Market getting closed and we had square of for the day, we can take a look at the holdings
holdings.getTotal()

0

In [16]:
# This will get the actions performed per candle
action_history = runner.getActionHistory()

In [17]:
len(action_history)

390

In [24]:
action_strings = list(map( lambda x:x.act.value,action_history))

In [ ]:
action_strings.count("buy") # number of times we bought

12

In [ ]:
action_strings.count("sell") # number of times we sold

12

In [ ]:
# Perhaps you want to store the entire simulation into a csv 
from Data.simulation_writer import writer as sim_writer

sim_writer("ANET_Workings", runner) # this will save the simulated results in a csv


In [28]:
# Perhaps you want to rank the actions taken for training ml models

from ML.action_ranker import RwrdRealizedReturnActionRanker

ranker = RwrdRealizedReturnActionRanker("ANET_Workings",updated_name="ANET_workings_ranked")

In [29]:
ranker.rankAction()